In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field
from typing import TypedDict, Annotated
from dotenv import load_dotenv
import operator

In [2]:
load_dotenv()

True

In [3]:
model = ChatGroq(model= "llama-3.1-8b-instant")

In [4]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description="Detailed feedback for the essay")
    score: int = Field(description="Score out of 10", ge=0, le=10)
    

In [5]:
structured_model=model.with_structured_output(EvaluationSchema)

In [6]:
essay="""
I still remember the afternoon I first watched ‘Ramayana: The Legend of Prince Rama’ 
directed by Yugo Sako — a film brought to life through an unlikely collaboration 
between Indian storytelling and Japanese animation. That memory stayed with me not 
just as a piece of childhood nostalgia, but as an early lesson in what becomes possible 
when two cultures decide to build something together. Growing up, I kept finding the 
same thread: my grandmother’s mechanical Citizen watch ticking faithfully on her wrist 
for decades, Toyota cars navigating the chaotic rhythms of Indian streets with quiet 
reliability. Japan was always there — precise, purposeful, and deeply committed to craft. 
That impression never left me. 
Today, as I complete my final year of computer engineering, I find myself at a point 
where that long-held admiration has a natural place to go. I have spent the last few years 
building real systems — not just studying them. My work spans artificial intelligence, 
full-stack development, and cloud architecture, and I have been deliberate about 
choosing projects that sit at the intersection of technical depth and practical value. The 
METI Government of Japan Internship Program feels like the right next step: a chance 
to stop admiring from a distance and start contributing directly. 
The experience that shaped my technical thinking most was my research internship, 
where I built an AI model using TinyML and 6LoWPAN for intelligent IoT systems. The 
constraint was the point — making machine learning work on devices with almost no 
computational headroom forced a kind of discipline I had not encountered before. Every 
decision had to be justified. That appreciation for efficiency, for doing more with less, is 
something I see mirrored in Japan’s approach to manufacturing and engineering. I 
would genuinely like to apply it in that context, particularly toward predictive 
maintenance or smart factory solutions where resource-conscious AI can make a real 
difference. 
At Gati AI Tech Innovations, I moved from research into production. I built 
context-aware intelligent agents, developed deep learning proof-of-concepts across 
multiple large language models, and constructed event-driven microservices using 
Kafka, Redis, and MongoDB to handle real-time data streams. These were not academic 
exercises — they were systems that had to work under load, handle failures gracefully, 
and integrate with existing infrastructure. That experience taught me how to think 
about software at a system level, not just a component level. 
Outside of my formal internships, I have built several independent platforms that reflect 
where I think technology is headed. GoEat is a MERN-stack food delivery platform with 
GeoJSON-based location filtering and AI-powered recommendations. Talent Talk is a 
Spring Boot freelancing platform with secure REST APIs and an AI-driven interview 
module. I also built a high-performance e-learning backend in GoLang and an 
Enterprise Multilingual RAG Chatbot using LangChain, vector databases, and LLM 
orchestration — designed specifically for businesses that need to query complex 
documents across languages. That last project is one I am particularly proud of, because 
it addresses a real friction point for companies trying to go global: information that 
exists in one language, and people who need it in another. 
I am aware that technical skill alone does not make someone a good colleague or a 
useful intern. Joining a Japanese company means joining a culture, and I take that 
seriously. As the co-founder and coordinator of Neuron, the AI Club at my university, I 
have learned how to run workshops, listen carefully, and help people with different 
backgrounds find common ground around a shared problem. I participate actively in 
cultural events at my university and in the broader community. I am not someone who 
needs a comfort zone to function — I find new environments genuinely interesting, not 
intimidating. 
What draws me most to this program is the philosophy behind Japanese engineering 
excellence. Monozukuri — the art of making things well — and Kaizen — the 
commitment to continuous, honest improvement — are not just management concepts I 
read about. They describe a way of working I want to internalize. I want to understand 
what it looks like in practice, inside a Japanese company, on a real project. I want to 
bring that standard back with me. 
In the longer arc, I see this internship as one concrete step toward the kind of 
India-Japan technological collaboration that both countries have committed to. That 
vision is large, but it is built from individual people who cross the distance, do good 
work, and carry something back. I would like to be one of those people. I am ready to 
contribute from day one, and I am genuinely excited about what this experience can 
become.
"""

In [7]:
prompt=f'Evaluate the language quality of the following essay and provide a feedback and assign score {essay}'

structured_model.invoke(prompt)

EvaluationSchema(feedback="The essay is well-structured, and the writer showcases a deep understanding of Japanese culture and engineering practices. The writer's passion for technology and collaboration is evident throughout the essay. However, some sentences are repetitive, and the writer could have provided more specific examples to support their claims. Overall, the essay is strong, and the writer demonstrates a clear and compelling vision for their future.", score=9)

In [10]:
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add ]
    avg_score: float

In [12]:
def evaluate_language(state: UPSCState):
    prompt=f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']} '
    output= structured_model.invoke(prompt)
    return {'language_feedback' : output.feedback, 'individual_score':[output.score]}

In [ ]:
graph= StateGraph(UPSCState)
graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_analysis',evaluate_analysis)
graph.add_node('evaluate_thought',evaluate_thought)
graph.add_node('final_evaluation',final_evaluation)